# RQ6: How does body type influence cargo volume, seating capacity, and price premium?

**Hypothesis:** SUVs and Trucks command a price premium over Sedans/Hatchbacks due to larger cargo and seating capacity.

**Methodology:**
1. Compare price, cargo volume, seating by body type
2. ANOVA across body types for each outcome
3. Scatter: cargo volume vs price, coloured by body type (PDF)
4. Summary statistics table (CSV)

In [ ]:
import pandas as pd, numpy as np, os, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import f_oneway
warnings.filterwarnings('ignore')
plt.rcParams.update({'font.family':'serif','font.size':11,'axes.titlesize':12,'axes.labelsize':11,'figure.dpi':300,'axes.spines.top':False,'axes.spines.right':False})

for p in ['/kaggle/input/electric-vehicle-market-and-pricing-dataset-2026/ev_market_2026.csv','ev_market_2026.csv']:
    if os.path.exists(p): df = pd.read_csv(p); break

df = df.dropna(subset=['body_type','price_usd','cargo_volume_cubic_ft','seating_capacity'])
print('Body types:', df['body_type'].value_counts().to_dict())

In [ ]:
body_types = df['body_type'].value_counts().index.tolist()
COLORS = {'SUV':'#e74c3c','Truck':'#e67e22','Sedan':'#3498db','Hatchback':'#2ecc71','Coupe':'#9b59b6','Van':'#1abc9c'}
MARKERS = {'SUV':'o','Truck':'s','Sedan':'^','Hatchback':'D','Coupe':'P','Van':'X'}

rows = []
for bt in body_types:
    sub = df[df['body_type']==bt]
    rows.append({'Body Type':bt,'N':len(sub),
        'Mean Price ($k)': round(sub['price_usd'].mean()/1000,1),
        'SD Price ($k)': round(sub['price_usd'].std()/1000,1),
        'Mean Cargo (ft³)': round(sub['cargo_volume_cubic_ft'].mean(),1),
        'SD Cargo (ft³)': round(sub['cargo_volume_cubic_ft'].std(),1),
        'Mean Seating': round(sub['seating_capacity'].mean(),1),
    })
tbl = pd.DataFrame(rows).sort_values('Mean Price ($k)', ascending=False)

# ANOVA
for col, label in [('price_usd','Price'),('cargo_volume_cubic_ft','Cargo'),('seating_capacity','Seating')]:
    F, p = f_oneway(*[df[df['body_type']==bt][col].values for bt in body_types])
    print(f'{label} ANOVA: F={F:.2f}, p={p:.4f}')
print(tbl.to_string(index=False))

In [ ]:
Fp, pp = f_oneway(*[df[df['body_type']==bt]['price_usd'].values for bt in body_types])
Fc, pc = f_oneway(*[df[df['body_type']==bt]['cargo_volume_cubic_ft'].values for bt in body_types])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Scatter: cargo vs price
for bt in body_types:
    sub = df[df['body_type']==bt]
    ax1.scatter(sub['cargo_volume_cubic_ft'], sub['price_usd']/1000,
                color=COLORS.get(bt,'#888'), marker=MARKERS.get(bt,'o'),
                alpha=0.45, s=28, label=bt)
ax1.set_xlabel('Cargo Volume (ft³)'); ax1.set_ylabel('Price ($1,000s)')
ax1.set_title('Cargo Volume vs. Price by Body Type', fontsize=11)
ax1.legend(fontsize=8, frameon=True, framealpha=0.9, ncol=2)

# Box: price by body type
sorted_bt = tbl.sort_values('Mean Price ($k)',ascending=False)['Body Type'].tolist()
data = [df[df['body_type']==bt]['price_usd'].values/1000 for bt in sorted_bt]
bp = ax2.boxplot(data, patch_artist=True, widths=0.6, medianprops={'color':'#333','linewidth':1.8}, whiskerprops={'linewidth':1.2}, capprops={'linewidth':1.2})
for patch, bt in zip(bp['boxes'], sorted_bt): patch.set_facecolor(COLORS.get(bt,'#888')); patch.set_alpha(0.7)
ax2.set_xticklabels(sorted_bt, rotation=25, ha='right')
ax2.set_ylabel('Price ($1,000s)')
ax2.set_title(f'Price Distribution by Body Type\n(F={Fp:.2f}, p={pp:.4f})', fontsize=11)

fig.suptitle('Body Type: Price, Cargo, and Seating Analysis', fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig('RQ6_BodyType_Price_Cargo.pdf', bbox_inches='tight', format='pdf')
plt.show(); print('Saved: RQ6_BodyType_Price_Cargo.pdf')

In [ ]:
tbl.to_csv('RQ6_BodyType_Summary_Table.csv', index=False)
print('Saved: RQ6_BodyType_Summary_Table.csv'); tbl